#Preparing data for Gold layer

We use Text Splitter Package from LangChain to split our document into chunks

In [0]:
%pip install --quiet --upgrade langchain_text_splitters
%restart_python

Lets make sure that all Silver Tables are still intact

In [0]:
# Set the current catalog
spark.catalog.setCurrentCatalog("workspace")

# Set the current schema (database)
spark.catalog.setCurrentDatabase("genai_course")

In [0]:
silver_df = spark.read.table("silver_docs_parsed")
display(silver_df)

Semantic Text Splitting

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 150

splitter = RecursiveCharacterTextSplitter (
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n ==page == \n", " ==page== ", "\n", "\n", " ", ""]
)

def chunk_text(text):
    chunks = []
    if isinstance (text, str) and text.strip():
        for c in splitter.split_text(text):
            if c and c.strip():
                chunks.append(c)
    return chunks


Test this by using one document from Silver Layer record

In [0]:
doc = """A black and white logo featuring a stylized letter D intersected by a diagonal line and flanked by curved signal waves. ## DerarTel MOBILE TECHNOLOGY # Product Specifications: dPhone D1 Lite *Status: Internal/Confidential* ## 1. Executive Summary The Phone D1 Lite is designed as a lighter, more accessible take on a premium smartphone, combining solid performance with a clean, simple design. It aims to compete with the iPhone lineup while focusing on open ecosystem support, flexible hardware, and strong, future‑ready security. ![A composite image showing a blue smartphone held in a hand and a disassembled phone with labeled internal components.](#) *Figure 1. dPhone D1 Lite product image showcasing the full design and internal features* ==page== ## 2. Key Value Propositions - The "Fluid" OS: A proprietary operating system built on a mikrokernel architecture that allows for 100% app compatibility with both Android and iOS environments. | Feature | Specification | |---------|---------------| | Processor | Helios G1L Neural Chip (6nm architecture) | | Display | 6.1" Super‑AMOLED "Infinity" Display | | Storage | 256GB | | Refresh Rate | Dynamic 1Hz to 60Hz | | Battery | 5,200 mAh with 20W Rapid Charging | | Security | Biometric "Vein‑Scan" (Under‑display) | | Connectivity | 5G Ready & Satellite SOS | ## 3. Environmental Resistance Designed for everyday use, the dPhone D1 Lite does not include water or dust resistance. No Ingress Protection supported; the device is not certified against dust or water exposure, so it should be kept away from moisture, liquids, and dusty environments to maintain optimal performance. - End of Document - """

chunks = chunk_text(doc)

print(f"Number of chunks: {len(chunks)}")

for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c}")
    print("-" * 80)


We need to define a pandas udf to apply the Python function to each row in our spark dataframe. Pandas UDF optimizes data transfer between spark and Python using Apache Airflow format.

In [0]:
from typing import Iterator

from pyspark.sql.functions import pandas_udf

import pandas as pd

@pandas_udf("array<string>")
def get_chunks(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    for s in batch_iter:
        yield s.apply(chunk_text)

Now we call this UDF within our spark dataframe

In [0]:
df_chunks = (silver_df.withColumn("chunk", get_chunks("plain_text"))
                   .select("path", "chunk"))

display(df_chunks)

For vector Store each row must have its own unique ID

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

df_chunks = df_chunks.withColumn("id", monotonically_increasing_id())

## display(df_chunks)

df_chunks.write.mode("overwrite").saveAsTable("gold_docs_chunks")

In [0]:
df_gold = spark.read.table("gold_docs_chunks")
display(df_gold)